# 07 · Fly + Mario: one whole run (looming only)

**Goal:** put everything together. The game from notebooks 05–06, the sense from notebook 03 and the brain from notebooks 02 and 04 now run in one loop, and you watch a complete episode.

You will learn
- what a **fly config** is (`configs/flies/*.yaml`) — attachments are *config*, not code
- the frame loop in `runner/session.py`: **objects → senses → brain → actions → buttons → game**
- how to step a run yourself and record anything you like (`FrameRecord`)
- why runs are **reproducible** (seed, hash) and how **rewind** works (`snapshot` / `restore`)

> Only `senses/`, `actions/` and `runner/` may know both the fly and Mario. This notebook uses the **runner**, the one door between them.

## 0. Data

In [ ]:
DATA_DIR = "~/fly_mario_fly_data"      # <-- change this to the folder that holds malecns_v1.0/

import numpy as np
import matplotlib.pyplot as plt

import _helpers as H
H.use_data_folder(DATA_DIR)

from fly_mario_fly.contracts import Scene
from fly_mario_fly.runner import Run, Options, load_fly, dump_fly, active_senses, active_actions, brain_spec, resolve_knobs

## 1. A fly is a YAML file

Which senses are attached to which objects, which actions are on, which dials are set. Nothing else. This is the *looming-only* fly: one sense, attached to the things that can hurt (or block) Mario.
`home` is a whole-fly sense that keeps the fly pulling to the right (without it a fly that sees nothing would just stand still).

In [ ]:
fly = load_fly("looming_only")
print(dump_fly(fly))

In [ ]:
print("senses :", [s.name for s in active_senses(fly)])
print("actions:", [a.name for a in active_actions(fly)])
print("brain  :", brain_spec(fly))
print("dials  :", {k: round(v, 4) for k, v in resolve_knobs(fly).items()})

`resolve_knobs` fills every dial the fly doesn't mention with its default. Whatever the yaml sets is what a person (or the evolution running in the background) chose.

## 2. One frame, in the runner's words

This is what `Run.step()` does each frame, in order (see `runner/session.py`):

1. read RAM: `read_mario`, and `find()` for every object plug-in → a list of `ObjectInstance` *(notebook 06)*
2. for each **sense**: give it the objects attached to it (or all of them, for a whole-fly sense) → `SenseOutput` *(notebook 03)*
3. **sum** the senses' inputs per neuron group, and `brain.advance` one frame *(notebook 04)*
4. for each **action**: read the group rates → a `ButtonIntent` *(notebook 04, section 5)*
5. resolve the intents into buttons, send them to the emulator, collect the result *(notebook 05)*

Nothing in step 3 knows about Mario, and nothing in step 1 knows about the fly. The runner is the only place that holds both.

## 3. Run it

A `Scene` is a reproducible situation: level, idle frames before the fly takes over (`noop`), sensor noise (`sigma`) and the **seed** for that noise.
We step the run ourselves and keep what we want from each `FrameRecord`.

In [ ]:
scene = Scene("1-1", noop=3, sigma=0.0, seed=101)
run = Run(fly, scene, Options(moment_every_s=0))        # no auto-saved moments: we only want the trace
run.start()

trace = {"x": [], "theta": [], "TTMn": [], "DNp01": [], "A": [], "right": [], "types": []}
keep = {}                                                 # a few records with pictures, for the filmstrip
while not run.finished:
    rec = run.step()
    trace["x"].append(rec.x)
    trace["theta"].append(rec.outputs["looming"].inputs.get("LPLC2_R", 0.0))
    trace["TTMn"].append(rec.rates["TTMn"]); trace["DNp01"].append(rec.rates["DNp01"])
    trace["A"].append("A" in rec.buttons); trace["right"].append("right" in rec.buttons)
    trace["types"].append(sorted(t for t, objs in rec.found.items() if objs))
    if len(trace["x"]) % 25 == 0:
        keep[len(trace["x"])] = rec

result = run.result()
print("outcome:", result.outcome, "| furthest x:", result.max_x, "| frames:", result.counts["frames"], "| hash:", result.hash[:12])

`outcome` is one of `flag` (reached the flagpole), `stuck`, `timeout` or a death. The **hash** is a fingerprint of the exact button sequence: if two runs have the same hash they played the same game.

## 4. Watch what the fly did

Top to bottom: Mario's progress; what the looming sense sent in; the brain's two output neurons; and when button A (jump) was held.

In [ ]:
t = np.arange(len(trace["x"]))
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(t, trace["x"], color="k"); axes[0].set_ylabel("Mario x (px)")
axes[1].plot(t, trace["theta"], color="tab:green"); axes[1].set_ylabel("LPLC2_R input (theta)")
axes[2].plot(t, trace["DNp01"], label="DNp01"); axes[2].plot(t, trace["TTMn"], label="TTMn"); axes[2].axhline(resolve_knobs(fly)["jump.ttmn_thresh"], ls="--", color="grey", label="jump threshold"); axes[2].legend(); axes[2].set_ylabel("rate")
axes[3].fill_between(t, np.array(trace["A"], dtype=float), step="mid", color="tab:red", label="A (jump)"); axes[3].set_yticks([]); axes[3].legend(loc="upper right")
axes[3].set_xlabel("frame (after the 3 idle frames)")
axes[0].set_title("one run of the looming-only fly on World 1-1, seed 101")
plt.tight_layout(); plt.show()

presses = np.flatnonzero(np.diff(np.array(trace["A"], dtype=int)) == 1)
print("number of separate jumps:", len(presses))

The jumps come right after the *theta* spikes: something is about to be hit, the Giant Fiber fires, TTMn crosses its threshold, A is held. Between obstacles theta is small and the `home` input keeps the fly walking right.

## 5. Look at the moments of a jump

The runner records the senses' drawing instructions too (`rec.outputs[...].overlay`). Here are the frames around the first jump, with the box the looming sense was looking at.

In [ ]:
first = int(presses[0]) + 1
want = [first - 12, first, first + 12, first + 24]

# re-run to those frames (deterministic!) and grab the pictures
run2 = Run(fly, scene, Options(moment_every_s=0)); run2.start()
shots = {}
while not run2.finished and len(shots) < len(want):
    rec = run2.step()
    if run2.frame - 1 in want:
        shots[run2.frame - 1] = rec

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, f in zip(axes, want):
    rec = shots[f]
    boxes = [o.geometry for o in rec.outputs["looming"].overlay if o.kind == "box"]
    H.show_frame(rec.screen, ax, title=f"frame {f}   A={'A' in rec.buttons}   TTMn={rec.rates['TTMn']:.2f}", boxes=boxes, scroll=rec.scroll_x_after)
plt.show()

Red box = the object the looming sense is currently reporting. Note frame *first − 12* (the object is approaching, A not yet pressed) and *first* (A pressed).

## 6. Reproducible by construction

Everything random is seeded (`Scene.seed` feeds `np.random.default_rng`); the emulator is deterministic. Run the same scene again and you should get the same hash. Add 5 % sensor noise with another seed and you get a (slightly) different run.

In [ ]:
def play(**kw):
    r = Run(fly, Scene("1-1", noop=3, **kw), Options(moment_every_s=0))
    res = r.play()
    return res.outcome, res.max_x, res.hash[:12]

print("seed 101, no noise  :", play(sigma=0.0, seed=101))
print("seed 101, no noise  :", play(sigma=0.0, seed=101), " <- same hash")
print("seed 7, noise 5 %   :", play(sigma=0.05, seed=7))
print("seed 8, noise 5 %   :", play(sigma=0.05, seed=8))

## 7. Rewind and retry

A run can be **snapshotted** at any frame: emulator state, brain state and the senses' memory. `run.restore(moment)` goes back. The tool's *moments* are exactly this (saved to disk).
Let's snapshot a bit before the first jump, play to the end, then rewind in a fresh run and play again: the second ending must be identical.

In [ ]:
r = Run(fly, scene, Options(moment_every_s=0)); r.start()
while r.frame < first - 15:
    r.step()
moment = r.snapshot("before the first jump")
print("snapshot taken at fly frame", r.frame, "| Mario x =", r.env.ram[0x86] + 256 * r.env.ram[0x6D])

r.play(); end1 = r.result()

r2 = Run(fly, scene, Options(moment_every_s=0)); r2.start()      # a fresh run = a fresh emulator, as the tool does for a retry
r2.restore(moment)                                               # rewind: game + brain + sense memory
r2.play(); end2 = r2.result()
print("first ending :", end1.outcome, end1.max_x, end1.hash[:12])
print("after rewind :", end2.outcome, end2.max_x, end2.hash[:12])

## 8. What does attaching the sense buy? Compare flies

Same scene, three flies: nothing attached (`blank`), the looming-only fly, and the *champion* from the repo (looming + hearing + smell + headroom, with evolved dials).

In [ ]:
for name in ["blank", "looming_only", "champion_e1s"]:
    f = load_fly(name)
    res = Run(f, scene, Options(moment_every_s=0)).play()
    print(f"{name:14s} senses attached: {sorted(f.attachments)!s:60s} -> {res.outcome:6s} max x = {res.max_x:.0f}")

The blank fly sees nothing, so it does nothing (it never takes a step beyond the start). The looming-only fly, with one sense and the real escape circuit, already gets Mario to the flag with this seed, and so does the champion.
With a single clean seed they look the same. Whether other seeds, idle-frame counts or sensor noise separate them is a good thing to test yourself (see *Try this*).

> Honesty check: this is an invented cue (θ computed from the NES) driving a real circuit (the MaleCNS escape pathway), plus an invented `home` input and invented thresholds. A win says the *combination* works.

## Recap

- A **fly** = YAML: attachments, actions, dials. A **scene** = level + seed + noise. A **run** = `Run(fly, scene)`.
- Each frame: objects → senses → (sum per group) → brain → rates → actions → buttons → emulator.
- Runs are deterministic and fingerprinted (hash); a snapshot lets you rewind game, brain and sense memory together.
- Evolution (`python -m fly_mario_fly.evolve`) and the web app (`python -m fly_mario_fly.api`) are just more users of the same `Run`.

**Try this**
1. Copy `configs/flies/looming_only.yaml` to a new file, remove `pit` from the attachments, and run it. Where does it die now?
2. Change `jump.ttmn_thresh` in your copy (`Run(load_fly("my_fly"), scene)`). What happens at 0.15? At 0.06?
3. Attach `looming` only to `goomba`. Which obstacles now stop Mario?
4. Compare `looming_only` and `champion_e1s` over several seeds with `sigma=0.05` and different `noop` values (section 6 shows how). Does one of them win more often?

**Where next:** `python -m fly_mario_fly.api` and the web app (`web/`) let you do all of this by clicking.